## Financial Market Data Pipeline

In [ ]:
!pip install requests
!pip install pandas
!pip install pyarrow
!pip install azure-storage-file-datalake



In [ ]:
!pip uninstall azure-storage-file-datalake azure-storage-blob azure-core -y
!pip install azure-storage-file-datalake

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
!{sys.executable} -m pip install azure-storage-file-datalake

In [ ]:
import requests
import time
import pandas as pd
from dotenv import load_dotenv
import os
from azure.storage.filedatalake import DataLakeServiceClient


# Extraction

In [ ]:
def transform_data(data, symbol):
    # Extract the time series dictionary
    time_series = data["Time Series (Daily)"]

    # Convert to DataFrame (keys become index)
    df = pd.DataFrame.from_dict(time_series, orient="index")

    # Reset index and convert to datetime
    df = df.reset_index().rename(columns={"index": "datetime"})
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Rename Alpha Vantage columns
    df = df.rename(columns={
        "1. open": "open",
        "2. high": "high",
        "3. low": "low",
        "4. close": "close",
        "5. volume": "volume"
    })

    # Convert datatypes
    df = df.astype({
        "open": float,
        "high": float,
        "low": float,
        "close": float,
        "volume": int
    })

    # Add symbol column
    df["symbol"] = symbol

    # Sort by date for correct feature engineering
    df = df.sort_values("datetime")

    # Feature engineering
    df["daily_return"] = df["close"].pct_change() * 100
    df["price_range"] = df["high"] - df["low"]
    df["prev_close"] = df["close"].shift(1)
    df["price_direction"] = (df["close"] > df["prev_close"]).astype(int)
    
    # fill null
    df["daily_return"] = df["daily_return"].fillna(0)
    df["prev_close"] = df["prev_close"].fillna(0)

    return df


In [ ]:
import time
import requests
import pandas as pd

def fetch_with_retry(url, max_retries=3):
    for attempt in range(max_retries):
        response = requests.get(url)
        data = response.json()

        # Success
        if "Time Series (Daily)" in data or "Name" in data:
            return data

        # Rate limit hit
        if "Information" in data:
            print(" Rate limit hit. Waiting 60 seconds...")
            time.sleep(60)
            continue

        # Unexpected error
        print(" Unexpected API response:", data)
        return None

    print("Max retries reached. Skipping.")
    return None


def fetch_data(symbols, api_key):
    price_frames = []
    overview_rows = []

    for symbol in symbols:

        # ---- PRICE DATA ----
        url_price = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={api_key}"
        data_price = fetch_with_retry(url_price)

        if data_price is None or "Time Series (Daily)" not in data_price:
            print(f" Failed price fetch for {symbol}")
            continue

        df_price = transform_data(data_price, symbol)
        price_frames.append(df_price)
        print(f"Price data fetched for {symbol}")

        # ---- OVERVIEW ----
        url_overview = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={api_key}"
        data_overview = fetch_with_retry(url_overview)

        if data_overview is None:
            print(f"Failed overview fetch for {symbol}")
            continue

        overview_rows.append({
            "symbol": symbol,
            "company_name": data_overview.get("Name"),
            "sector": data_overview.get("Sector"),
            "exchange": data_overview.get("Exchange")
        })
        print(f" Overview fetched for {symbol}")

    df_prices = pd.concat(price_frames).reset_index(drop=True)
    df_overview = pd.DataFrame(overview_rows)

    return df_prices, df_overview


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()


symbols = ["AAPL", "MSFT", "GOOGL", "AMZN"]

df_prices, df_overview = fetch_data(symbols, api_key)

print("Prices:")
display(df_prices.head())

print("Overview:")
display(df_overview.head())


In [ ]:
api_key = "YOUR_KEY_HERE"
symbols = ["IBM", "MSFT", "AMZN", "META"]

#run_date = pd.Timestamp.today().normalize()

df_prices, df_overview = fetch_data(symbols, api_key, run_date)

df_final = df_prices.merge(df_overview, on="symbol")

print(df_final.head())


In [ ]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd

load_dotenv()
api_key = os.getenv("API_KEY")

print("API KEY:", api_key)


def fetch_data(symbol, api_key):
    print("USING API KEY:", api_key)  # debug
    url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    if data.get('status') != 'ok':
        print(f"error with {symbol}: {data.get('message')}")
    return data


def transform_data(data):
    df = pd.DataFrame(data['values'])
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int'
    })
    return df


symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
all_data = []

for symbol in symbols:
    data = fetch_data(symbol, api_key)
    print("RAW RESPONSE:", data)  # debug
    df = transform_data(data)
    df['symbol'] = symbol
    all_data.append(df)

final_df = pd.concat(all_data).reset_index(drop=True)
final_df.head()


In [ ]:
https://api.twelvedata.com/time_series?symbol=AAPL&interval=1min&apikey=0bb27de9424a464ba47cd89911e640de&outputsize=4


In [ ]:
final_df.head()

In [ ]:
from dotenv import load_dotenv, dotenv_values
values = dotenv_values()
print(values)